# Before vs After Fine-tuning: Wait-k & Full Decoding

Compares **BLEU** and **Average Lagging (AL)** for the Telugu→English model:
- **Base** (`sarvamai/sarvam-translate`, no fine-tuning)
- **Fine-tuned** (wait-k LoRA adapter from `train.py`)

under **full-sentence** decoding and **wait-k** decoding for several `k`.

Reuses the same code paths as training/eval (`src/waitk.py`, `src/load.py`, `evaluate.py`)
so the prompt format and policy match exactly. Models are loaded one at a time to fit on a single 16 GB GPU.

## Configuration

In [8]:
BASE_MODEL  = "sarvamai/sarvam-translate"
# Fine-tuned weights: either a Lightning .ckpt (mid-training best) or a PEFT
# adapter dir (checkpoints/final from train.py). Auto-detected in the loader cell.
ADAPTER     = "/home/ramesh/IASNLP/SIM_MT/waitk_finetune/checkpoints/waitk-1-1010-0.797.ckpt"
DATASET     = "../in22_conv_te_en.json"  # IN22-style json: input=Telugu, reference=English
TGT_LANG    = "English"

K_VALUES    = [1, 3, 5, 7]               # wait-k policies to sweep
N_SUBSET    = 10                        # sentences for the (slow) wait-k eval; None = all
FULL_BATCH  = 4                          # batch size for full-sentence decoding
TOKENIZE    = "intl"                     # sacrebleu tokenizer (intl matches IN22)
OUT_JSON    = "eval_compare_results.json"

In [2]:
import gc, json
import torch, sacrebleu, pandas as pd
from tqdm.auto import tqdm

from src.load import load_model, load_from_ckpt
from src.waitk import load_pairs, wait_k_decode, average_lagging
from evaluate import translate_full, score   # reuse the exact eval helpers


def load_finetuned(path):
    """Load either a Lightning .ckpt or a PEFT adapter directory."""
    if path.endswith(".ckpt"):
        return load_from_ckpt(path)
    return load_model(BASE_MODEL, adapter_path=path)


sources, references = load_pairs(DATASET)
if N_SUBSET:
    sources, references = sources[:N_SUBSET], references[:N_SUBSET]
print(f"Evaluating on {len(sources)} sentences from {DATASET}")

/home/ramesh/IASNLP/SiLLM/MT_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluating on 10 sentences from ../in22_conv_te_en.json


## Evaluation routine
Runs full-sentence + every wait-k policy for one loaded model and returns a list of result rows.

In [3]:
def run_all_policies(model, tokenizer, tag):
    rows = []

    # --- Full-sentence (offline upper bound) ---
    hyps = translate_full(model, tokenizer, sources, TGT_LANG, FULL_BATCH)
    bleu, chrf = score(hyps, references, TOKENIZE)
    rows.append({"model": tag, "policy": "full", "k": None,
                 "bleu": bleu, "chrf": chrf, "AL": float("nan")})
    print(f"[{tag}] full     BLEU={bleu:5.2f}  chrF++={chrf:5.2f}")

    # --- Wait-k sweep ---
    for k in K_VALUES:
        hyps, als = [], []
        for src in tqdm(sources, desc=f"[{tag}] wait-{k}", leave=False):
            hyp, trace = wait_k_decode(model, tokenizer, src, k=k,
                                       target_language=TGT_LANG, verbose=False)
            hyps.append(hyp)
            als.append(average_lagging(trace, len(src.split())))
        bleu, chrf = score(hyps, references, TOKENIZE)
        mean_al = sum(als) / len(als)
        rows.append({"model": tag, "policy": f"wait-{k}", "k": k,
                     "bleu": bleu, "chrf": chrf, "AL": mean_al})
        print(f"[{tag}] wait-{k:<2} BLEU={bleu:5.2f}  chrF++={chrf:5.2f}  AL={mean_al:5.2f}")
    return rows


def free(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

## Evaluate the base model
Loaded and freed before the fine-tuned model so only one ~4B model sits on the GPU at a time.

In [4]:
model, tokenizer = load_model(BASE_MODEL, adapter_path=None)
base_rows = run_all_policies(model, tokenizer, tag="base")
free(model)

full: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]


[base] full     BLEU=29.63  chrF++=53.50


[base] wait-1  BLEU=19.62  chrF++=46.33  AL= 1.71


[base] wait-3  BLEU=27.42  chrF++=49.43  AL= 3.10


[base] wait-5  BLEU=28.35  chrF++=51.56  AL= 3.85


[base] wait-7  BLEU=29.63  chrF++=53.50  AL= 4.00


## Evaluate the fine-tuned model

In [9]:
free(model)
model, tokenizer = load_finetuned(ADAPTER)
ft_rows = run_all_policies(model, tokenizer, tag="finetuned")
free(model)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 18.62it/s]


trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 15.64 GiB of which 2.38 MiB is free. Process 2392120 has 674.00 MiB memory in use. Including non-PyTorch memory, this process has 14.95 GiB memory in use. Of the allocated memory 14.72 GiB is allocated by PyTorch, and 37.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Comparison table

In [ ]:
df = pd.DataFrame(base_rows + ft_rows)

# Side-by-side: BLEU and AL before vs after, per policy.
pivot = df.pivot_table(index="policy", columns="model", values=["bleu", "AL"])
pivot[("bleu", "Δ")] = pivot[("bleu", "finetuned")] - pivot[("bleu", "base")]
# Order rows: full first, then wait-k ascending.
order = ["full"] + [f"wait-{k}" for k in K_VALUES]
pivot = pivot.reindex([p for p in order if p in pivot.index])
pivot.round(2)

In [ ]:
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump({"dataset": DATASET, "n": len(sources), "tokenize": TOKENIZE,
               "rows": base_rows + ft_rows}, f, indent=2)
df.to_csv("eval_compare_results.csv", index=False)
print(f"Saved {OUT_JSON} and eval_compare_results.csv")

## Quality–latency curve (BLEU vs AL)
Each point is a wait-k policy; dashed horizontal lines are the full-sentence ceilings.
A good fine-tune shifts the wait-k curve **up** (higher BLEU) and/or **left** (lower AL).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
for tag, color in [("base", "tab:gray"), ("finetuned", "tab:blue")]:
    sub = df[(df.model == tag) & (df.policy != "full")].sort_values("AL")
    plt.plot(sub.AL, sub.bleu, "o-", color=color, label=f"{tag} (wait-k)")
    for _, r in sub.iterrows():
        plt.annotate(f"k={int(r.k)}", (r.AL, r.bleu),
                     textcoords="offset points", xytext=(5, 4), fontsize=8, color=color)
    full_bleu = df[(df.model == tag) & (df.policy == "full")].bleu.iloc[0]
    plt.axhline(full_bleu, ls="--", color=color, alpha=0.6,
                label=f"{tag} full = {full_bleu:.1f}")

plt.xlabel("Average Lagging  (lower = more simultaneous)")
plt.ylabel(f"BLEU ({TOKENIZE})")
plt.title(f"Wait-k quality–latency: base vs fine-tuned\n({DATASET}, n={len(sources)})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("eval_compare_quality_latency.png", dpi=120)
plt.show()
print("Saved eval_compare_quality_latency.png")